## Ideal Pose Extractor
Extracts raw variable-length keypoints per clip directly from source videos,
uses the impact model to find the exact contact frame, slices the three phases
from the *raw* sequence, then builds k-means clusters per phase per shot.

**Why raw sequences matter:** the 100-frame uniform-sampled arrays in
`processed/X.npy` compress every clip to the same length first — so any
phase boundary mapped back onto them lands at nearly the same index for every
clip, giving identical shapes. Working from the raw video ensures each phase
has its genuine duration before clustering.

In [10]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import mediapipe as mp
from torchvision import transforms
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
from PIL import Image
from collections import defaultdict

In [11]:
LABEL_NAMES = [
    'Cover Drive',
    'Cut',
    'Flick',
    'Forward Defence',
    'Pull',
    'Sweep',
]

# DATA_DIR  = "/home/azureuser/kabuni-aiml-folder/Shot-Classifcation/New_Trimmed_for_Shot"
DATA_DIR  = r"D:\CA_POC\dataset"
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cpu


In [12]:
# ── Impact model ──────────────────────────────────────────────────────────

# IMPACT_CKPT        = '/home/azureuser/kabuni-aiml-folder/ball-bat-impact-mobilenetv2/impact_cls_550_data.pt'
IMPACT_CKPT        = r"D:\CA_POC\impact_cls_550_data.pt"
IMPACT_CONF_THRESH = 0.70

def build_impact_model(device):
    m = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
    for idx, block in enumerate(m.features):
        for p in block.parameters():
            p.requires_grad = idx >= 12
    m.classifier = nn.Sequential(
        nn.Dropout(0.3), nn.Linear(1280, 128), nn.ReLU(),
        nn.Dropout(0.2), nn.Linear(128, 2),
    )
    return m.to(device)

ckpt               = torch.load(IMPACT_CKPT, map_location=DEVICE)
impact_class_names = ckpt['class_names']
impact_img_size    = tuple(ckpt['image_size'])
impact_cls_idx     = impact_class_names.index('impact')

impact_model = build_impact_model(DEVICE)
impact_model.load_state_dict(ckpt['model_state_dict'])
impact_model.eval()

impact_transform = transforms.Compose([
    transforms.Resize(impact_img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=ckpt['mean'], std=ckpt['std']),
])

def predict_impact_conf(bgr_frame):
    """Returns confidence that this frame contains ball-bat impact."""
    rgb    = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    tensor = impact_transform(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(impact_model(tensor), dim=1)[0]
    return probs[impact_cls_idx].item()

print('Impact model ready')

Impact model ready


In [13]:
# ── MediaPipe pose ────────────────────────────────────────────────────────

mp_pose = mp.solutions.pose
pose    = mp_pose.Pose(static_image_mode=False)

def extract_kp_flat(bgr_frame):
    """Returns (99,) keypoint vector or None if pose not detected."""
    res = pose.process(cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB))
    if not res.pose_landmarks:
        return None
    return np.array([[lm.x, lm.y, lm.z] for lm in res.pose_landmarks.landmark]).flatten()

MP = dict(L_SHOULDER=11, R_SHOULDER=12, L_ELBOW=13, R_ELBOW=14,
          L_WRIST=15,    R_WRIST=16,    L_HIP=23,   R_HIP=24,
          L_KNEE=25,     R_KNEE=26,     L_ANKLE=27, R_ANKLE=28, NOSE=0)

In [14]:
# ── Feature builder (matches feature-engineering.ipynb exactly) ───────────

def calculate_angle(a, b, c):
    ba = a - b; bc = c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

def normalize_pose(seq):
    """seq: (T, 99) → (T, 99) hip-centred, torso-scaled."""
    s = seq.copy().reshape(-1, 33, 3)
    for i, kp in enumerate(s):
        hip  = (kp[MP['L_HIP']] + kp[MP['R_HIP']]) / 2
        kp  -= hip
        kp  /= (np.linalg.norm(kp[MP['L_SHOULDER']] - kp[MP['L_HIP']]) + 1e-6)
        s[i] = kp
    return s.reshape(-1, 99)

def mirror_if_left(seq):
    """Mirror left-handed batter to canonical right-handed frame."""
    s = seq.reshape(-1, 33, 3)
    mid = len(s) // 2
    w = s[max(0, mid-5): mid+5]
    if np.mean(w[:, MP['R_WRIST'], 1]) < np.mean(w[:, MP['L_WRIST'], 1]):
        s[:, :, 0] = -s[:, :, 0]
        for l, r in [(11,12),(13,14),(15,16),(23,24),(25,26),(27,28)]:
            s[:, [l, r]] = s[:, [r, l]]
    return s.reshape(-1, 99)

def build_features_seq(seq):
    """seq: (T, 99) normalised → (T, 214) full feature vector."""
    def ext_angles(sq):
        out = []
        for kp in sq.reshape(-1, 33, 3):
            out.append([
                calculate_angle(kp[MP['L_SHOULDER']], kp[MP['L_ELBOW']], kp[MP['L_WRIST']]),
                calculate_angle(kp[MP['R_SHOULDER']], kp[MP['R_ELBOW']], kp[MP['R_WRIST']]),
                calculate_angle(kp[MP['L_HIP']],      kp[MP['L_KNEE']], kp[MP['L_ANKLE']]),
                calculate_angle(kp[MP['R_HIP']],      kp[MP['R_KNEE']], kp[MP['R_ANKLE']]),
                calculate_angle(kp[MP['L_KNEE']], kp[MP['L_ANKLE']], kp[MP['L_ANKLE']] + np.array([0.1,0,0])),
                calculate_angle(kp[MP['R_KNEE']], kp[MP['R_ANKLE']], kp[MP['R_ANKLE']] + np.array([0.1,0,0])),
            ])
        return np.array(out)

    def ext_cricket(sq):
        out = []
        for kp in sq.reshape(-1, 33, 3):
            bat_vec    = kp[MP['R_WRIST']] - kp[MP['R_ELBOW']]
            hip_w      = abs(kp[MP['L_HIP'], 0] - kp[MP['R_HIP'], 0]) + 1e-6
            body_vec   = kp[MP['R_SHOULDER']] - kp[MP['R_HIP']]
            hip_y      = (kp[MP['L_HIP'], 1] + kp[MP['R_HIP'], 1]) / 2
            sh_y       = (kp[MP['L_SHOULDER'], 1] + kp[MP['R_SHOULDER'], 1]) / 2
            torso      = abs(sh_y - hip_y) + 1e-6
            hip_vec    = kp[MP['R_HIP']] - kp[MP['L_HIP']]
            sh_vec     = kp[MP['R_SHOULDER']] - kp[MP['L_SHOULDER']]
            out.append([
                np.degrees(np.arctan2(bat_vec[1], bat_vec[0])),
                (kp[MP['R_ANKLE'], 0] - kp[MP['L_ANKLE'], 0]) / hip_w,
                np.degrees(np.arctan2(body_vec[1], body_vec[0])),
                abs(kp[MP['NOSE'], 0] - kp[MP['R_KNEE'], 0]),
                (hip_y - kp[MP['R_WRIST'], 1]) / torso,
                np.degrees(np.arctan2(hip_vec[2], hip_vec[0])),
                np.degrees(np.arctan2(sh_vec[2],  sh_vec[0])),
                calculate_angle(kp[MP['R_HIP']], kp[MP['R_KNEE']], kp[MP['R_ANKLE']]),
                np.linalg.norm(kp[MP['L_WRIST']] - kp[MP['R_WRIST']]),
                kp[MP['NOSE'], 1] - sh_y,
            ])
        return np.array(out)

    # Velocity: np.diff needs at least 2 frames.
    # If segment is a single frame, duplicate it so diff gives zeros.
    if len(seq) < 2:
        seq = np.vstack([seq, seq])
    v = np.diff(seq, axis=0)
    v = np.vstack([v, v[-1]])
    f = np.concatenate([seq, v, ext_angles(seq), ext_cricket(seq)], axis=1)
    assert f.shape[1] == 214
    return f

In [15]:
# ── Phase config ──────────────────────────────────────────────────────────

PHASE_OFFSETS = {
    'backswing':      (-20, -3),   # up to 18 frames before contact
    'impact':         ( -2,  2),   # 5 frames around contact (always shortest)
    'follow_through': (  3, 20),   # up to 18 frames after contact
}
K_CLUSTERS = 3

def wrist_velocity_peak(kp_seq):
    """Fallback impact index: frame of max wrist speed."""
    wrists = (kp_seq[:, MP['L_WRIST']*3 : MP['L_WRIST']*3+3] +
              kp_seq[:, MP['R_WRIST']*3 : MP['R_WRIST']*3+3]) / 2
    speed  = np.linalg.norm(np.diff(wrists, axis=0), axis=1)
    return int(np.argmax(speed)) + 1

def slice_phase_raw(kp_seq, impact_idx, phase):
    """
    Returns the raw (variable-length) slice for a phase.
    kp_seq: (T, 99) — raw valid keypoints, NOT resampled.
    impact_idx: index into kp_seq of the impact frame.
    """
    lo, hi  = PHASE_OFFSETS[phase]
    T       = len(kp_seq)
    start   = max(0, impact_idx + lo)
    end     = min(T, impact_idx + hi + 1)

    # Widen the window until we have at least 2 frames (needed for np.diff).
    extra = 0
    while (end - start) < 2 and extra < T:
        extra += 1
        start  = max(0, impact_idx + lo - extra)
        end    = min(T, impact_idx + hi + extra + 1)

    seg = kp_seq[start:end]

    # Last resort: use the whole clip
    if len(seg) < 2:
        seg = kp_seq

    return seg

In [16]:
# ── Per-clip extraction: raw keypoints + impact frame ─────────────────────
# One-time offline step. Saves raw_clips.npy so it can be reloaded.
# Each entry: {'label': int, 'kp_raw': (T_raw, 99), 'impact_idx': int}
#
# This is the ONLY place where phase durations vary between clips.
# Typical values at 30 fps:
#   backswing      ~ 20–30 frames
#   impact         ~  5–9  frames
#   follow_through ~ 20–35 frames

# RAW_CLIPS_PATH = '/home/azureuser/kabuni-aiml-folder/Shot-Classifcation/Pose Estimation engine/processed/raw_clips.npy'
RAW_CLIPS_PATH = r"D:\CA_POC\new_processed\raw_clips.npy"

labels    = [l for l in sorted(os.listdir(DATA_DIR)) if l != 'Backward Defence']
label_map = {label: i for i, label in enumerate(labels)}

if os.path.exists(RAW_CLIPS_PATH):
    raw_clips = np.load(RAW_CLIPS_PATH, allow_pickle=True).tolist()
    print(f'Loaded {len(raw_clips)} cached raw clips.')
else:
    raw_clips = []
    for label in labels:
        folder = os.path.join(DATA_DIR, label)
        files  = sorted(os.listdir(folder))
        print(f'Processing {label} ({len(files)} clips)...')

        for fname in files:
            vpath = os.path.join(folder, fname)
            cap   = cv2.VideoCapture(vpath)
            if not cap.isOpened():
                continue

            kp_list        = []     # valid-frame keypoints
            impact_confs   = []     # (valid_idx, conf) for frames above threshold

            valid_idx = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break

                # Impact model
                conf = predict_impact_conf(frame)
                if conf >= IMPACT_CONF_THRESH:
                    impact_confs.append((valid_idx, conf))

                # Pose
                kp = extract_kp_flat(frame)
                if kp is not None:
                    kp_list.append(kp)
                    valid_idx += 1

            cap.release()

            if len(kp_list) < 10:
                continue

            kp_seq = np.array(kp_list)   # (T_raw, 99)  — genuinely variable length

            # Impact index in valid-kp space
            if impact_confs:
                impact_idx = max(impact_confs, key=lambda x: x[1])[0]
                impact_idx = min(impact_idx, len(kp_seq) - 1)
            else:
                impact_idx = wrist_velocity_peak(kp_seq)

            raw_clips.append({
                'label':      label_map[label],
                'kp_raw':     kp_seq,
                'impact_idx': impact_idx,
            })

    np.save(RAW_CLIPS_PATH, raw_clips, allow_pickle=True)
    print(f'Saved {len(raw_clips)} raw clips.')

# Print phase duration statistics
phase_lens = defaultdict(list)
for clip in raw_clips:
    T = len(clip['kp_raw']); imp = clip['impact_idx']
    for ph in PHASE_OFFSETS:
    
        lo, hi = PHASE_OFFSETS[ph]
        phase_lens[ph].append(min(T, imp+hi+1) - max(0, imp+lo))

print('\nPhase duration stats (raw frames):')
for ph in PHASE_OFFSETS:
    arr = np.array(phase_lens[ph])
    print(f'  {ph:20s}  mean={arr.mean():.1f}  min={arr.min()}  max={arr.max()}')

Processing Cover Drive (86 clips)...
Processing Cut (85 clips)...
Processing Flick (89 clips)...
Processing Forward Defense (86 clips)...
Processing Pull (87 clips)...
Processing Sweep (77 clips)...
Saved 488 raw clips.

Phase duration stats (raw frames):
  backswing             mean=11.4  min=-2  max=18
  impact                mean=4.8  min=3  max=5
  follow_through        mean=9.9  min=-2  max=18


In [17]:
# ── Build ideal pose clusters from raw phases ─────────────────────────────
# Steps per shot per phase:
#   1. Slice the raw variable-length phase segment from each clip.
#   2. Normalise + mirror to canonical frame.
#   3. Build features (214-dim) from the raw segment.
#   4. Flatten to (N, T_phase * F) — T_phase differs per clip, so we
#      normalise each segment to the MEDIAN phase length for that shot/phase.
#      This preserves meaningful duration differences while enabling k-means.
#   5. Select the 20 most representative clips.
#   6. K-means (k=3) on the flattened segments.

def resample_to(seg, n):
    if len(seg) == 0:
        return np.zeros((n, seg.shape[-1]))
    idx = np.linspace(0, len(seg)-1, n).astype(int)
    return seg[idx]

# Group by label
by_label = defaultdict(list)
for clip in raw_clips:
    by_label[clip['label']].append(clip)

def select_representative(seqs_flat, seqs_orig, top_k=20):
    """Select top_k most central sequences by cosine similarity."""
    norms  = np.linalg.norm(seqs_flat, axis=1, keepdims=True) + 1e-6
    unit   = seqs_flat / norms
    scores = (unit @ unit.T).sum(axis=1) - 1.0
    idx    = np.argsort(scores)[::-1][:top_k]
    return seqs_flat[idx], [seqs_orig[i] for i in idx]

def kmeans_centroids(data, k=3, n_iter=60, seed=42):
    if len(data) < k * 2:
        return np.median(data, axis=0, keepdims=True)
    rng = np.random.default_rng(seed)
    c   = data[rng.choice(len(data), k, replace=False)].copy()
    for _ in range(n_iter):
        labels = np.argmin(np.linalg.norm(data[:, None] - c[None], axis=2), axis=1)
        new_c  = np.array([data[labels==i].mean(0) if (labels==i).any() else c[i] for i in range(k)])
        if np.allclose(c, new_c, atol=1e-7): break
        c = new_c
    return c

ideal_poses     = {}
ideal_poses_raw = {}

for label_idx, label_name in enumerate(LABEL_NAMES):
    clips = by_label[label_idx]
    print(f'\n{label_name}  ({len(clips)} clips)')

    ideal_poses[label_idx]     = {}
    ideal_poses_raw[label_idx] = {}

    for phase in ('backswing', 'impact', 'follow_through'):
        # Step 1: slice raw phase segments from every clip
        raw_segs  = []    # list of (T_phase_i, 99)  variable length
        feat_segs = []    # list of (T_phase_i, 214) variable length

        for clip in clips:
            kp_raw = clip['kp_raw']                              # (T_raw, 99)
            seg    = slice_phase_raw(kp_raw, clip['impact_idx'], phase)
            seg_n  = normalize_pose(seg)                         # normalise
            seg_n  = mirror_if_left(seg_n)                       # canonical frame
            raw_segs.append(seg_n)
            feat_segs.append(build_features_seq(seg_n))

        # Step 2: compute MEDIAN phase length across all clips for this shot/phase
        median_len = int(np.median([len(s) for s in raw_segs]))
        median_len = max(median_len, 3)   # at least 3 frames

        # Step 3: resample every segment to median_len
        raw_resampled  = np.array([resample_to(s, median_len) for s in raw_segs])   # (N, T_med, 99)
        feat_resampled = np.array([resample_to(s, median_len) for s in feat_segs])  # (N, T_med, 214)

        # Step 4: representative selection + k-means
        flat_f = feat_resampled.reshape(len(feat_resampled), -1)
        flat_r = raw_resampled.reshape(len(raw_resampled), -1)

        top_flat_f, _ = select_representative(flat_f, feat_segs, top_k=min(20, len(feat_segs)))
        top_flat_r, _ = select_representative(flat_r, raw_segs,  top_k=min(20, len(raw_segs)))

        c_f = kmeans_centroids(top_flat_f, k=K_CLUSTERS)   # (K, T_med*214)
        c_r = kmeans_centroids(top_flat_r, k=K_CLUSTERS)   # (K, T_med*99)

        ideal_poses[label_idx][phase]     = c_f.reshape(-1, median_len, 214)
        ideal_poses_raw[label_idx][phase] = c_r.reshape(-1, median_len, 99)

        print(f'  {phase:20s}  median_len={median_len:3d}  '
              f'centroids={ideal_poses[label_idx][phase].shape}')

np.save(r"D:\CA_POC\new_processed\ideal_poses.npy",     ideal_poses,     allow_pickle=True)
np.save(r"D:\CA_POC\new_processed\ideal_poses_raw.npy", ideal_poses_raw, allow_pickle=True)
print('\nIdeal poses saved.')


Cover Drive  (85 clips)
  backswing             median_len= 18  centroids=(3, 18, 214)
  impact                median_len=  5  centroids=(3, 5, 214)
  follow_through        median_len= 18  centroids=(3, 18, 214)

Cut  (80 clips)
  backswing             median_len=  9  centroids=(3, 9, 214)
  impact                median_len=  5  centroids=(3, 5, 214)
  follow_through        median_len=  7  centroids=(3, 7, 214)

Flick  (78 clips)
  backswing             median_len= 18  centroids=(3, 18, 214)
  impact                median_len=  5  centroids=(3, 5, 214)
  follow_through        median_len= 13  centroids=(3, 13, 214)

Forward Defence  (83 clips)
  backswing             median_len=  9  centroids=(3, 9, 214)
  impact                median_len=  5  centroids=(3, 5, 214)
  follow_through        median_len= 10  centroids=(3, 10, 214)

Pull  (85 clips)
  backswing             median_len=  9  centroids=(3, 9, 214)
  impact                median_len=  5  centroids=(3, 5, 214)
  follow_through   

In [18]:
# ── Sanity check — shapes should differ across phases AND shots ────────────

loaded = np.load(r'D:\CA_POC\new_processed\ideal_poses.npy', allow_pickle=True).item()
shapes = {}
for shot_idx, phases in loaded.items():
    for phase, arr in phases.items():
        key = f'{LABEL_NAMES[shot_idx]:20s} / {phase:20s}'
        shapes[key] = arr.shape
        print(f'  {key}: {arr.shape}')

# Verify that impact phase is shorter than backswing / follow_through
print('\nPhase length check (expect impact < backswing and follow_through):')
for shot_idx in loaded:
    bs  = loaded[shot_idx]['backswing'].shape[1]
    imp = loaded[shot_idx]['impact'].shape[1]
    ft  = loaded[shot_idx]['follow_through'].shape[1]
    ok  = imp <= bs and imp <= ft
    print(f'  {LABEL_NAMES[shot_idx]:20s}  bs={bs:3d}  imp={imp:3d}  ft={ft:3d}  {"✓" if ok else "✗ CHECK"}')

  Cover Drive          / backswing           : (3, 18, 214)
  Cover Drive          / impact              : (3, 5, 214)
  Cover Drive          / follow_through      : (3, 18, 214)
  Cut                  / backswing           : (3, 9, 214)
  Cut                  / impact              : (3, 5, 214)
  Cut                  / follow_through      : (3, 7, 214)
  Flick                / backswing           : (3, 18, 214)
  Flick                / impact              : (3, 5, 214)
  Flick                / follow_through      : (3, 13, 214)
  Forward Defence      / backswing           : (3, 9, 214)
  Forward Defence      / impact              : (3, 5, 214)
  Forward Defence      / follow_through      : (3, 10, 214)
  Pull                 / backswing           : (3, 9, 214)
  Pull                 / impact              : (3, 5, 214)
  Pull                 / follow_through      : (3, 9, 214)
  Sweep                / backswing           : (3, 9, 214)
  Sweep                / impact              : (3, 